# 🏢 [Lab 4] LangChain Middleware 기반 에이전틱 RAG Self-Correction & 2계층 평가 하네스 실무

> **과정명**: 기업 데이터 연동을 위한 에이전틱 RAG 아키텍처 구축 실무 (Day 1 - Module 5)  
> **핵심 주제**: `LangChain 1.0+ AgentMiddleware`, `2-Stage Self-Correction (wrap_tool_call + after_agent)`, `2-Tier Evaluation Harness (Process Trajectory + RAGAS Outcome)`

---

## 🗺️ 엔터프라이즈 미들웨어 하네스 & 2계층 평가 아키텍처

AI 에이전트를 프로덕션에 배포할 때 가장 큰 문제는 **"도구 검색 실패 시 맹목적으로 포기하거나 환각을 일으키는 것(Retry Waste & Hallucination)"**과 **"에이전트가 어떤 근거와 경로를 거쳐 답을 도출했는지 감사(Audit)할 수 없는 것"**입니다.

본 실습에서는 에이전트의 내부 그래프를 난잡하게 하드코딩하는 대신, **LangChain 공식 `AgentMiddleware` 표준 스택**을 활용하여:
1. **도구 레벨 (`wrap_tool_call`)**: 검색 실패 시 쿼리를 실시간 정규화하여 1회 자동 재시도하는 `QueryRewriter`
2. **응답 레벨 (`after_agent`)**: 사실성(Groundedness) 루브릭 검증 실패 시 피드백 메시지를 주입하여 자가 수정하는 `HallucinationGrader`
3. **2계층 평가 하네스 (`before/after_agent`)**: 과정(Tool Trajectory: 도구선택, 쿼리품질, 궤적효율)과 결과(RAGAS 4대 지표)를 통합 감사하는 전사 평가 시스템을 구축합니다.

```mermaid
flowchart TB
    subgraph Client ["사용자 / 클라이언트"]
        Query["사용자 질문"]
    end

    subgraph CoreAgent ["🤖 ReAct Core Agent (create_agent)"]
        LLM["Gemini 3.5 Flash"]
        Tools["RAG 도구셋 (BOK Reports, Policy, Knowledge Graph)"]
    end

    subgraph MiddlewareHarness ["🛡️ Production RAG Middleware Harness Stack"]
        direction TB
        
        subgraph ToolMiddleware ["1. 도구 레벨 자가 수정 & 궤적 캡처 (wrap_tool_call)"]
            T1["Tool 격발 가로채기"] --> T2{"검색 품질 검사<br>(결과 공백 / 저유사도)"}
            T2 -- "미달" --> TRewrite["QueryRewriter<br>(동의어/맥락 보강 1회 재검색)"]
            TRewrite --> TExec["도구 재실행"]
            T2 -- "정상" --> TExec
            TExec --> TLog["도구 호출 궤적(Trajectory) 로깅"]
        end

        subgraph RespMiddleware ["2. 응답 레벨 자가 수정 (after_agent)"]
            R1["최종 생성 응답 가로채기"] --> R2{"환각/문서근거 채점<br>(HallucinationGrader)"}
            R2 -- "환각 감지 / 근거 부족" --> RFeedback["🛑 [Self-Correction Blocking Error]<br>오류 원인 피드백 메시지 주입<br>(최대 2회 자가 수정 루프)"]
            RFeedback --> LLM
            R2 -- "통과 (Grounded)" --> RPass["최종 응답 승인"]
        end

        subgraph EvalMiddleware ["3. 2계층 평가 하네스 (RAGEvalHarnessMiddleware)"]
            direction LR
            EvalTraj["[과정 평가: Trajectory]<br>• 도구 선택 정확도<br>• 인자 적절성 / 쿼리 품질<br>• 궤적 효율성 (Step Economy)<br>• 도구 결과 활용률"]
            EvalOutcome["[결과 평가: RAGAS]<br>• Faithfulness (환각 0%)<br>• Answer Relevance<br>• Context Precision<br>• Context Recall"]
        end
    end

    Query --> CoreAgent
    CoreAgent <--> ToolMiddleware
    CoreAgent <--> RespMiddleware
    RespMiddleware --> EvalMiddleware
    EvalMiddleware --> AuditLog["📊 세션 감사 로그 & 정량 평가 리포트"]
```


## 1. 환경 설정 및 베이스라인 에이전트/도구셋 로드

먼저 필요한 라이브러리를 로드하고 환경 변수를 확인합니다.
앞서 Lab 1과 Lab 2에서 설계했던 **3대 엔터프라이즈 도구(사내 규정, 한국은행 산업 보고서, 조직도 지식그래프)**를 정의합니다.


In [ ]:
import os
import sys
import time
import json
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents.middleware import AgentMiddleware, wrap_tool_call

# 프로젝트 루트 경로 추가 및 .env 로드
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
load_dotenv(os.path.join(PROJECT_ROOT, ".env"), override=True)

from app.utils.llm import get_llm
from app.utils.context import AgentContext

# 메인 모델 및 심판 모델 셋업
llm = get_llm(model_name="gemini-3.5-flash", temperature=0.0)
judge_llm = get_llm(model_name="gemini-3.5-flash", temperature=0.0)

print(f"✅ 환경 초기화 완료! (LLM: {llm.model_name})")


### 1.1 3대 엔터프라이즈 RAG 도구 정의 (Controlled Test Doubles)
본 실습에서는 앞선 1~2교시에서 구축한 4대 DB(사내 규정 Chroma, BOK 보고서 Chroma, NetworkX 조직 그래프, GraphRAG)의 도메인 특성을 그대로 반영한 3개의 전문 도구를 바인딩합니다:
1. `query_company_policy`: 사내 여비, 복무, 보안 규정 데이터베이스
2. `query_bok_reports`: 한국은행 2024년 분기별 반도체/이차전지/자동차 거시 보고서
3. `query_org_graph`: 사내 부서, 팀장, 결재 승인자, 프로젝트 배정 지식 그래프

> [!NOTE]
> **💡 [하네스 엔지니어링 설계 의도: 왜 제어된 도구(Controlled Tool)를 사용하나요?]**
> * **평가 대상의 순수 격리 (Isolation)**: 4교시의 유일한 학습 목표는 **"LangChain `AgentMiddleware`가 도구 쿼리를 자율 재작성하고, 환각을 감지하여 자가 수정(Self-Correction) 피드백 루프를 완벽히 통제하는가"**입니다.
> * **100% 재현 가능한 벤치마크 (Deterministic Evaluation)**: 실제 Vector DB의 유사도 오차나 네트워크 지연 등 외부 노이즈를 배제하고, 구어체 검색 실패나 허위 규정 환각 시나리오를 결정론적으로 재현하기 위함입니다.
> *(※ 실제 서비스 배포 코드는 `app/agents/corrective_rag_agent.py`에서 실제 `app/database/` 디스크 DB들과 연결되어 동작합니다.)*


In [ ]:
@tool
def query_company_policy(query: str) -> str:
    """
    사내 규정집(출장/여비, 복무, 정보보안, 인사/보수 등)을 정밀 검색합니다.
    사내 정책, 지원금 한도, 출장비, 보안 규칙, 결재 절차에 대한 질문에 사용하세요.
    """
    q_lower = query.lower()
    if "출장" in q_lower or "숙박" in q_lower or "항공" in q_lower or "일비" in q_lower:
        return (
            "[사내 여비/출장 규정 제14조]\n"
            "1. 해외 출장 숙박비: A등급 지역($250/일), B등급 지역($200/일), C등급 지역($150/일) 실비 상한 지원.\n"
            "2. 항공권 이용 기준: 임원 및 수석 이상은 비즈니스석 탑승, 책임/선임/전임은 이코노미석 탑승이 원칙임.\n"
            "3. 국내 출장 일비: 1일당 30,000원 정액 지급 (식비 25,000원 별도 실비 청구 가능)."
        )
    elif ("법인카드" in q_lower or "법인 카드" in q_lower) and ("소명" in q_lower or "사유서" in q_lower or "지침" in q_lower or "절차" in q_lower or "사용" in q_lower):
        return (
            "[법인카드 관리 지침 제8조 (주말/휴일 사용 통제)]\n"
            "주말 및 공휴일 법인카드 사용은 원칙적으로 금지됩니다. "
            "불가피한 사업상 목적으로 사용한 경우, 결제일로부터 3영업일 이내에 사내 ERP에 "
            "'주말 사용 사유서'와 증빙 자료(참석자 명단, 업무 회의록 등)를 등록하고 소속 부서장의 사후 결재를 받아야 합니다."
        )
    elif "보안" in q_lower or "chatgpt" in q_lower or ("생성형" in q_lower and "ai" in q_lower):
        return (
            "[정보보안 관리 규정 제22조 (생성형 AI 사용 지침)]\n"
            "1. 임직원은 외부 생성형 AI 서비스에 고객 개인정보, 미공개 소스코드, 핵심 재무정보 입력을 엄격히 금지함.\n"
            "2. 위반 행위 적발 시 보안위원회 심의를 거쳐 경고, 감봉, 정직 또는 징계 해고 등 사규에 따른 중징계 처분을 받음."
        )
    else:
        return "사내 규정 데이터베이스에서 해당 키워드와 관련된 유효 규정을 찾지 못했습니다."

@tool
def query_bok_reports(query: str) -> str:
    """
    한국은행 주요 산업(반도체, 이차전지, 자동차, 철강 등) 분기별 경제 동향 보고서를 검색합니다.
    거시 경제 전망, 글로벌 수요/공급망 리스크, 산업별 수출 실적 및 정책 대응에 대한 질문에 사용하세요.
    """
    q_lower = query.lower()
    if "반도체" in q_lower or "hbm" in q_lower or "파운드리" in q_lower:
        return (
            "[한국은행 2024년 4분기 주요 산업 동향 - 반도체 부문]\n"
            "1. HBM 메모리 전망: AI 데이터센터 투자 가속화로 글로벌 HBM 수요는 전년 동기 대비 150% 이상 폭증.\n"
            "2. 주요 공급망 리스크: TSMC CoWoS 등 첨단 어드밴스드 패키징 병목 및 차세대 1b nm 수율 확보 지연.\n"
            "3. 수출 및 가동률: 하반기 반도체 수출은 전년비 28% 증가했으며, 레거시 파운드리 가동률은 2025년 상반기 80% 회복 전망."
        )
    elif "이차전지" in q_lower or "완성차" in q_lower or "ira" in q_lower or "자동차" in q_lower:
        return (
            "[한국은행 2024년 3분기 산업 리포트 - 자동차/배터리]\n"
            "1. IRA 대응 전략: 국내 완성차 업계는 북미 현지 상업용 리스/렌터카 판매 비중을 확대하여 보조금 공백을 방어함.\n"
            "2. 이차전지 합작공장: 북미 주요 OEM(GM, 포드)과의 배터리 합작공장(JV) 조기 완공 및 FTA 체결국 중심 광물 소싱 다변화 추진."
        )
    else:
        return "한국은행 산업 보고서에서 해당 내용에 대한 분석 자료를 찾을 수 없습니다."

@tool
def query_org_graph(query: str) -> str:
    """
    사내 조직도, 부서 구성원, 직속 결재선(상사), 투입된 프로젝트 관계 지식 그래프(Knowledge Graph)를 탐색합니다.
    인물 검색, 조직 관계, 결재 승인자, 프로젝트 참여 인력 확인에 사용하세요.
    """
    q_lower = query.lower()
    if "김철수" in q_lower or "클라우드운영팀" in q_lower:
        return (
            "[사내 지식 그래프 Subgraph]\n"
            "• 엔티티: 김철수 (직급: 팀장, 소속: 클라우드사업본부 클라우드운영팀)\n"
            "• 직속 상사(결재권자): 최동식 본부장 (직급: 상무, 소속: 클라우드사업본부)\n"
            "• 참여 프로젝트: '엔터프라이즈 하이브리드 클라우드 전환(Cloud-Next)' (역할: 총괄 PM)"
        )
    elif "박영희" in q_lower or "ai솔루션" in q_lower:
        return (
            "[사내 지식 그래프 Subgraph]\n"
            "• 엔티티: 박영희 (직급: 수석, 소속: AI솔루션본부 AI솔루션팀)\n"
            "• 직속 상사(결재권자): 이민호 본부장 (직급: 상무, 소속: AI솔루션본부)\n"
            "• 참여 프로젝트: '대규모 언어모델 에이전트 플랫폼 구축' (역할: 테크 리드)"
        )
    elif "sf-2025" in q_lower or "스마트팩토리" in q_lower:
        return (
            "[사내 지식 그래프 Subgraph]\n"
            "• 프로젝트: 스마트팩토리 구축 프로젝트 (SF-2025)\n"
            "• 총괄 책임자: 정우성 수석 (소속: 제조DX팀, 직급: 부장급 수석)\n"
            "• 투입 인력 소속 부서: 제조DX팀(5명), 클라우드운영팀(2명), 보안엔지니어링팀(2명)"
        )
    else:
        return "지식 그래프에서 해당 엔티티 및 관계를 조회할 수 없습니다."

tools = [query_company_policy, query_bok_reports, query_org_graph]
print(f"✅ 3대 엔터프라이즈 도구 바인딩 완료: {[t.name for t in tools]}")


## 2. [문제 제기] Naive ReAct 에이전트의 2대 취약점

미들웨어 하네스가 없는 순수 Naive ReAct 에이전트를 먼저 생성하여 실행해 봅니다.

```
[Naive ReAct의 한계]
1. 모호한 비격식 질문 ➔ 키워드 매칭 실패 ➔ 맹목적 포기 (Retry Waste)
2. 사내에 없는 허위 규정 질문 ➔ 모델 가중치(Parametric Memory)로 그럴듯하게 날조 (Hallucination)
```


In [ ]:
# 순수 Naive ReAct 에이전트 (미들웨어 없음)
naive_agent = create_agent(
    model=llm,
    tools=tools,
    checkpointer=MemorySaver(),
    middleware=[],  # 미들웨어 없음
    context_schema=AgentContext
)

print("🧪 Naive ReAct Agent 생성 완료!")


### 2.1 [테스트 1] 구어체/모호한 쿼리로 인한 검색 실패 사례
사용자가 *"그 뭐냐 주말에 법카 긁었을 때 소명하는 거 어떻게 해야 됨?"* 이라고 질문했을 때, Naive ReAct는 원본 구어체('법카 긁었을 때')를 그대로 검색하여 공백 결과를 받고 포기하거나 부정확한 답변을 냅니다.


In [ ]:
config = {"configurable": {"thread_id": "naive_test_1"}}
res1 = naive_agent.invoke(
    {"messages": [{"role": "user", "content": "그 뭐냐 주말에 법카 긁었을 때 소명하는 거 어떻게 해야 됨?"}]},
    config=config
)

print("🤖 [Naive Agent 답변]:\n", res1["messages"][-1].content)


### 2.2 [테스트 2] 사내에 없는 허위 규정에 대한 환각(Hallucination) 유도 사례
사내에 존재하지 않는 *"2026년 신규 도입된 우주 항공 개발 특별 연구 상여금 지급 기준과 금액"*을 질문했을 때, Naive Agent는 '규정에 없다'고 선을 긋지 못하고 그럴듯한 가상의 수치를 지어냅니다.


In [ ]:
config = {"configurable": {"thread_id": "naive_test_2"}}
res2 = naive_agent.invoke(
    {"messages": [{"role": "user", "content": "2026년 신규 도입된 '우주 항공 개발 특별 연구 상여금'의 지급 대상과 1인당 최대 지급 금액은 얼마인가요?"}]},
    config=config
)

print("🤖 [Naive Agent 답변]:\n", res2["messages"][-1].content)


## 3. [Part 1] 도구 레벨 자가 수정 미들웨어 (`RAGToolCorrectionMiddleware`)

`@wrap_tool_call` 훅을 활용하여:
1. **중복 도구 호출 방지 (Anti-Spinning Cache)**: 동일한 쿼리로 헛도는 무한 루프 차단
2. **공백 결과 감지 및 자동 재시도 (Auto-Retry)**: 검색 결과가 비어있을 때 `QueryRewriter`를 호출하여 쿼리를 공식 용어로 정규화한 후 즉시 1회 재검색


In [ ]:
class RewrittenQuery(BaseModel):
    original_query: str = Field(description="원래 검색 쿼리")
    is_ambiguous: bool = Field(description="쿼리가 모호하거나 비격식체/오탈자가 있는가?")
    expanded_keywords: List[str] = Field(description="추가된 핵심 동의어 및 비즈니스 키워드")
    rewritten_query: str = Field(description="정규화되고 명확해진 최종 검색 쿼리")

class RAGToolCorrectionMiddleware(AgentMiddleware):
    def __init__(self, llm: Optional[Any] = None, enable_auto_retry: bool = True, verbose: bool = True):
        self.llm = llm
        self.enable_auto_retry = enable_auto_retry
        self.verbose = verbose
        self.seen_tool_calls = set()
        self.tool_trajectory_logs = []

    def rewrite_query(self, original_query: str, domain_hint: str = "general") -> str:
        structured_llm = self.llm.with_structured_output(RewrittenQuery)
        prompt = (
            f"당신은 엔터프라이즈 RAG 검색 쿼리 최적화기(Query Rewriter)입니다.\n"
            f"사용자의 원본 검색 쿼리를 분석하여 표준 비즈니스 용어와 핵심 키워드로 정규화하세요.\n\n"
            f"- 도메인 힌트: {domain_hint}\n"
            f"- 원본 쿼리: {original_query}\n"
        )
        try:
            result: RewrittenQuery = structured_llm.invoke([HumanMessage(content=prompt)])
            if self.verbose:
                print(f"🔄 [QueryRewriter] '{original_query}' ➔ '{result.rewritten_query}'")
            return result.rewritten_query
        except Exception as e:
            return original_query

    def _is_empty_result(self, result: Any) -> bool:
        if result is None: return True
        res_str = str(result).strip()
        empty_signals = ["결과를 찾지 못했습니다", "찾을 수 없습니다", "조회할 수 없습니다", "[]", "{}"]
        return any(sig in res_str for sig in empty_signals)

    def wrap_tool_call(self, request, handler):
        tool_name = request.tool_call.get("name", "unknown_tool") if hasattr(request, "tool_call") else getattr(request, "name", "unknown_tool")
        tool_args = request.tool_call.get("args", {}) if hasattr(request, "tool_call") else {}

        # 1. 중복 호출 검사
        sig = f"{tool_name}:{str(sorted(tool_args.items()))}"
        is_dup = sig in self.seen_tool_calls
        self.seen_tool_calls.add(sig)

        start_t = time.time()
        if self.verbose:
            print(f"🔧 [ToolInterceptor] ➡️ 도구 격발: {tool_name}({tool_args})" + (" [⚠️ 중복]" if is_dup else ""))

        # 1차 실행
        response = handler(request)
        dur_ms = int((time.time() - start_t) * 1000)

        # 2. 공백 시 자가 수정 1회 재시도
        retry_done = False
        if self._is_empty_result(response) and self.enable_auto_retry:
            q_key = next((k for k in ["query", "question", "search_query"] if k in tool_args), None)
            if q_key:
                orig_q = tool_args[q_key]
                rewritten_q = self.rewrite_query(orig_q, domain_hint=tool_name)
                if rewritten_q != orig_q:
                    new_args = dict(tool_args)
                    new_args[q_key] = rewritten_q
                    request.tool_call["args"] = new_args
                    
                    retry_t = time.time()
                    retry_resp = handler(request)
                    if not self._is_empty_result(retry_resp):
                        response = retry_resp
                        retry_done = True
                        dur_ms += int((time.time() - retry_t) * 1000)
                        if self.verbose:
                            print(f"✅ [ToolInterceptor] 자가 수정 재검색 성공!")

        self.tool_trajectory_logs.append({
            "tool_name": tool_name,
            "args": tool_args,
            "latency_ms": dur_ms,
            "is_duplicate": is_dup,
            "retry_done": retry_done,
            "status": "SUCCESS" if not self._is_empty_result(response) else "EMPTY"
        })
        return response

print("✅ RAGToolCorrectionMiddleware 정의 완료!")


## 4. [Part 2] 응답 레벨 환각 검증 & 자율 반성 미들웨어 (`RAGSelfCorrectionMiddleware`)

`after_agent` 라이프사이클 훅을 활용하여:
1. 최종 생성 답변이 검색된 **`ToolMessage`(Context)에만 엄격하게 사실적으로 근거(Groundedness)**하는지 루브릭 채점
2. 환각(Hallucination)이 감지되면 `🛑 [Self-Correction Blocking Error]` 피드백 메시지를 주입하여 에이전트 스스로 재답변을 생성하도록 유도


In [ ]:
class GroundednessEvaluation(BaseModel):
    is_grounded: bool = Field(description="답변의 내용이 검색된 Context에 사실적으로 근거하는가?")
    has_hallucination: bool = Field(description="Context에 없는 허위/가공된 사실이 포함되어 있는가?")
    groundedness_score: float = Field(description="사실 일치도 점수 (0.0 ~ 1.0)")
    critique_feedback: str = Field(description="에이전트가 답변을 수정할 수 있도록 제공하는 피드백")

class RAGSelfCorrectionMiddleware(AgentMiddleware):
    def __init__(self, judge_llm: Any, min_score: float = 0.70, max_retries: int = 2, verbose: bool = True):
        self.judge_llm = judge_llm
        self.min_score = min_score
        self.max_retries = max_retries
        self.verbose = verbose
        self._retry_counts: Dict[str, int] = {}

    def evaluate_groundedness(self, user_query: str, contexts: List[str], answer: str) -> GroundednessEvaluation:
        structured = self.judge_llm.with_structured_output(GroundednessEvaluation)
        context_str = "\n---\n".join(contexts) if contexts else "[검색된 문서 없음]"
        prompt = (
            f"당신은 엔터프라이즈 RAG 환각 전문 감사관입니다.\n\n"
            f"[사용자 질문]: {user_query}\n"
            f"[검색된 Context]:\n{context_str}\n\n"
            f"[에이전트 답변]:\n{answer}\n\n"
            f"답변이 Context에 사실적으로 근거하는지 평가하고 점수를 0.0~1.0으로 부여하세요. "
            f"만약 문서에 없는 질문에 대해 '사내 규정에 정보가 없다'고 올바르게 답변했다면 is_grounded=True 입니다."
        )
        return structured.invoke([HumanMessage(content=prompt)])

    def after_agent(self, state: dict, runtime=None) -> dict | None:
        messages = state.get("messages", [])
        if not messages or not isinstance(messages[-1], AIMessage):
            return None

        answer_text = str(messages[-1].content)
        user_query = next((str(m.content) for m in messages if isinstance(m, HumanMessage) and not str(m.content).startswith("🛑")), "")
        contexts = [str(m.content) for m in messages if isinstance(m, ToolMessage)]

        session_id = getattr(runtime, "session_id", "default_session") if runtime else "default_session"
        retries = self._retry_counts.get(session_id, 0)

        eval_res = self.evaluate_groundedness(user_query, contexts, answer_text)
        if self.verbose:
            icon = "✅" if (eval_res.is_grounded and eval_res.groundedness_score >= self.min_score) else "🔴"
            print(f"\n{icon} [GroundednessCheck] 점수: {eval_res.groundedness_score:.2f} (환각 여부: {eval_res.has_hallucination})")

        is_failed = (not eval_res.is_grounded) or (eval_res.groundedness_score < self.min_score) or eval_res.has_hallucination

        if is_failed and retries < self.max_retries:
            self._retry_counts[session_id] = retries + 1
            feedback_msg = (
                f"🛑 [Self-Correction Blocking Error: 사실성/환각 검증 실패]\n"
                f"검색된 문서에 근거하지 않은 가공된 사실이 감지되었습니다.\n"
                f"📌 피드백: {eval_res.critique_feedback}\n"
                f"오직 검색된 문서 내용에만 기반하여 다시 답변하세요. 문서에 없다면 '규정에 존재하지 않는다'고 명확히 밝히세요."
            )
            if self.verbose:
                print(f"🔄 [SelfCorrection] 자율 반성 피드백 주입 (재시도 {self._retry_counts[session_id]}/{self.max_retries})")
            return {"messages": list(messages) + [HumanMessage(content=feedback_msg)]}

        if session_id in self._retry_counts:
            del self._retry_counts[session_id]
        return None

print("✅ RAGSelfCorrectionMiddleware 정의 완료!")


## 5. [Part 3] 과정(Trajectory) & 결과(RAGAS) 2계층 평가 하네스 (`RAGEvalHarnessMiddleware`)

실무 에이전트 평가에서는 **결과(정답률)**뿐만 아니라 **과정(도구 선택 정확도, 쿼리 품질, 불필요한 낭비 여부)**을 동시에 감사해야 합니다.

| 계층 | 지표 | 측정 내용 |
| :--- | :--- | :--- |
| **과정 (Trajectory)** | `tool_selection_accuracy` | 질문 도메인에 적합한 도구를 골랐는가? |
| | `argument_quality_score` | 검색 인자(쿼리)의 키워드 적절성 |
| | `step_economy_score` | 중복 호출 없이 최소 스텝으로 도달했는가? (Anti-Spinning) |
| | `observation_grounding` | 도구 결과를 답변에 실질적으로 인용했는가? |
| **결과 (RAGAS)** | `faithfulness` | 답변의 사실 일치도 (환각 0% 검증) |
| | `answer_relevance` | 사용자 질문과의 부합도 |
| | `context_precision` | 검색된 문서 중 유효 정보 비율 |
| | `context_recall` | Ground Truth 핵심 사실 포함률 |


In [ ]:
class TrajectoryEvaluation(BaseModel):
    tool_selection_accuracy: float = Field(description="도구 선택 정확도 (0.0 ~ 1.0)")
    argument_quality_score: float = Field(description="쿼리 품질 (0.0 ~ 1.0)")
    step_economy_score: float = Field(description="궤적 효율성 (0.0 ~ 1.0)")
    observation_grounding_score: float = Field(description="도구 관측 활용도 (0.0 ~ 1.0)")
    trajectory_score: float = Field(description="과정 종합 점수 (0.0 ~ 1.0)")

class OutcomeEvaluation(BaseModel):
    faithfulness: float = Field(description="충실도 (0.0 ~ 1.0)")
    answer_relevance: float = Field(description="답변 관련성 (0.0 ~ 1.0)")
    context_precision: float = Field(description="컨텍스트 정밀도 (0.0 ~ 1.0)")
    context_recall: float = Field(description="컨텍스트 재현율 (0.0 ~ 1.0)")
    outcome_score: float = Field(description="결과 종합 점수 (0.0 ~ 1.0)")

class RAGEvalHarnessMiddleware(AgentMiddleware):
    def __init__(self, judge_llm: Any, verbose: bool = True):
        self.judge_llm = judge_llm
        self.verbose = verbose
        self.start_t = 0.0
        self.traj = []
        self.last_eval = {}

    def before_agent(self, state: dict, runtime=None):
        self.start_t = time.time()
        self.traj = []
        return None

    def wrap_tool_call(self, request, handler):
        t_name = request.tool_call.get("name", "tool") if hasattr(request, "tool_call") else "tool"
        t_args = request.tool_call.get("args", {}) if hasattr(request, "tool_call") else {}
        t0 = time.time()
        resp = handler(request)
        self.traj.append({"tool": t_name, "args": t_args, "latency_ms": int((time.time() - t0)*1000)})
        return resp

    def after_agent(self, state: dict, runtime=None):
        dur_ms = int((time.time() - self.start_t)*1000)
        messages = state.get("messages", [])
        if not messages: return None
        
        user_q = next((str(m.content) for m in messages if isinstance(m, HumanMessage) and not str(m.content).startswith("🛑")), "")
        answer = str(messages[-1].content)
        contexts = [str(m.content) for m in messages if isinstance(m, ToolMessage)]
        ground_truth = getattr(runtime, "ground_truth", "") if runtime else ""

        # 1. 과정 평가
        traj_prompt = f"질문: {user_q}\n실행 궤적: {json.dumps(self.traj, ensure_ascii=False)}\n답변: {answer}\n도구 선택 정확도, 쿼리 품질, 궤적 효율성(낭비시 감점), 도구 결과 인용도를 0.0~1.0 점수로 채점하세요."
        traj_eval: TrajectoryEvaluation = self.judge_llm.with_structured_output(TrajectoryEvaluation).invoke([HumanMessage(content=traj_prompt)])

        # 2. 결과 평가
        outcome_prompt = f"질문: {user_q}\n정답(Ground Truth): {ground_truth}\nContexts: {' '.join(contexts)}\n답변: {answer}\nRAGAS 4대 지표(faithfulness, answer_relevance, context_precision, context_recall)를 0.0~1.0으로 채점하세요."
        outcome_eval: OutcomeEvaluation = self.judge_llm.with_structured_output(OutcomeEvaluation).invoke([HumanMessage(content=outcome_prompt)])

        composite = round(0.4 * traj_eval.trajectory_score + 0.6 * outcome_eval.outcome_score, 3)
        self.last_eval = {
            "duration_ms": dur_ms,
            "tool_count": len(self.traj),
            "trajectory_score": traj_eval.trajectory_score,
            "faithfulness": outcome_eval.faithfulness,
            "answer_relevance": outcome_eval.answer_relevance,
            "context_precision": outcome_eval.context_precision,
            "context_recall": outcome_eval.context_recall,
            "outcome_score": outcome_eval.outcome_score,
            "composite_score": composite
        }
        if self.verbose:
            print(f"📊 [EvalHarness] Composite: {composite:.3f} | Trajectory: {traj_eval.trajectory_score:.2f} | Outcome: {outcome_eval.outcome_score:.2f} | 소요: {dur_ms}ms")
        return None

print("✅ RAGEvalHarnessMiddleware 정의 완료!")


## 6. [Part 4] 실전 벤치마크 대결: Naive Agent vs Middleware-Harnessed Agent

이제 10개의 엔터프라이즈 골든 데이터셋(`data/eval/golden_eval_dataset.json`)을 대상으로:
1. **Naive ReAct Agent (미들웨어 없음)**
2. **Middleware-Harnessed RAG Agent (도구 자가수정 + 응답 환각검증 + 평가하네스)**
두 시스템의 성능을 일괄 배치 실행하고 정량 비교합니다.


In [ ]:
# 1. 골든 데이터셋 로드
eval_dataset_path = os.path.join(PROJECT_ROOT, "data", "eval", "golden_eval_dataset.json")
with open(eval_dataset_path, "r", encoding="utf-8") as f:
    golden_dataset = json.load(f)

print(f"📂 골든 평가 데이터셋 {len(golden_dataset)}개 문항 로드 완료!")
for item in golden_dataset[:3]:
    print(f"  • [{item['id']}] {item['question']}")


### 6.1 Middleware-Harnessed RAG 에이전트 조립


In [ ]:
# 2. 미들웨어 인스턴스화
tool_correction_mw = RAGToolCorrectionMiddleware(llm=llm, enable_auto_retry=True, verbose=False)
self_correction_mw = RAGSelfCorrectionMiddleware(judge_llm=judge_llm, min_score=0.70, max_retries=2, verbose=False)
eval_harness_mw = RAGEvalHarnessMiddleware(judge_llm=judge_llm, verbose=False)

# 3. Harnessed Agent 구축
harnessed_agent = create_agent(
    model=llm,
    tools=tools,
    checkpointer=MemorySaver(),
    middleware=[tool_correction_mw, self_correction_mw, eval_harness_mw],
    context_schema=AgentContext
)

print("🛡️ Middleware-Harnessed Agent 조립 완료!")


### 6.2 10개 문항 일괄 벤치마크 실행 및 트레이스 수집


In [ ]:
benchmark_records = []

print("🚀 10개 골든 케이스 벤치마크 평가 시작...\n")

for item in golden_dataset:
    q_id = item["id"]
    query = item["question"]
    gt = item["ground_truth"]
    
    print(f"👉 평가 진행 중: [{q_id}] {query[:35]}...")
    
    # ── A. Naive Agent 실행 및 평가 ──
    cfg_naive = {"configurable": {"thread_id": f"bench_naive_{q_id}"}}
    t0 = time.time()
    resp_naive = naive_agent.invoke({"messages": [{"role": "user", "content": query}]}, config=cfg_naive)
    naive_dur = int((time.time() - t0) * 1000)
    naive_ans = str(resp_naive["messages"][-1].content)
    
    # Naive 결과 채점
    naive_eval_mw = RAGEvalHarnessMiddleware(judge_llm=judge_llm, verbose=False)
    class DummyRuntime: ground_truth = gt
    naive_eval_mw.before_agent({})
    # 도구 호출 추출
    for m in resp_naive["messages"]:
        if isinstance(m, ToolMessage):
            naive_eval_mw.traj.append({"tool": getattr(m, "name", "tool"), "args": {}, "latency_ms": 100})
    naive_eval_mw.after_agent({"messages": resp_naive["messages"]}, runtime=DummyRuntime())
    n_res = naive_eval_mw.last_eval

    # ── B. Harnessed Agent 실행 및 평가 ──
    cfg_harnessed = {"configurable": {"thread_id": f"bench_harness_{q_id}"}}
    class HarnessRuntime: ground_truth = gt
    resp_harnessed = harnessed_agent.invoke({"messages": [{"role": "user", "content": query}]}, config=cfg_harnessed)
    h_res = eval_harness_mw.last_eval

    benchmark_records.append({
        "ID": q_id,
        "Domain": item["domain"],
        "Naive_Faithfulness": n_res.get("faithfulness", 0.0),
        "Harness_Faithfulness": h_res.get("faithfulness", 0.0),
        "Naive_Relevance": n_res.get("answer_relevance", 0.0),
        "Harness_Relevance": h_res.get("answer_relevance", 0.0),
        "Naive_TrajScore": n_res.get("trajectory_score", 0.0),
        "Harness_TrajScore": h_res.get("trajectory_score", 0.0),
        "Naive_Composite": n_res.get("composite_score", 0.0),
        "Harness_Composite": h_res.get("composite_score", 0.0),
        "Naive_Latency_ms": naive_dur,
        "Harness_Latency_ms": h_res.get("duration_ms", 0),
    })

df_results = pd.DataFrame(benchmark_records)
print("\n🎉 10개 문항 벤치마크 완료!")


### 6.3 정량 평가 종합 데이터프레임 및 통계 분석


In [ ]:
# 요약 통계 테이블 출력
display_cols = ["ID", "Domain", "Naive_Faithfulness", "Harness_Faithfulness", "Naive_Composite", "Harness_Composite", "Naive_Latency_ms", "Harness_Latency_ms"]
print("📊 [벤치마크 결과 상세표]")
print(df_results[display_cols].to_markdown(index=False))

# 평균 지표 비교
summary_stats = pd.DataFrame({
    "지표 (Metric)": ["Faithfulness (충실도)", "Answer Relevance (관련성)", "Trajectory Score (과정 점수)", "Composite Total (종합 점수)", "Avg Latency (ms)"],
    "Naive ReAct": [
        df_results["Naive_Faithfulness"].mean(),
        df_results["Naive_Relevance"].mean(),
        df_results["Naive_TrajScore"].mean(),
        df_results["Naive_Composite"].mean(),
        df_results["Naive_Latency_ms"].mean(),
    ],
    "Middleware Harnessed": [
        df_results["Harness_Faithfulness"].mean(),
        df_results["Harness_Relevance"].mean(),
        df_results["Harness_TrajScore"].mean(),
        df_results["Harness_Composite"].mean(),
        df_results["Harness_Latency_ms"].mean(),
    ]
})

print("\n🏆 [최종 종합 평균 성적표]")
print(summary_stats.to_markdown(index=False))


### 6.4 📊 정량 비교 시각화 차트 생성 (Matplotlib)


In [ ]:
plt.figure(figsize=(12, 5))

# 1. Comparison of 4 Quality & Trajectory Metrics (Bar Chart)
plt.subplot(1, 2, 1)
metrics = ["Faithfulness", "Relevance", "Trajectory", "Composite"]
naive_scores = [
    df_results["Naive_Faithfulness"].mean(),
    df_results["Naive_Relevance"].mean(),
    df_results["Naive_TrajScore"].mean(),
    df_results["Naive_Composite"].mean()
]
harness_scores = [
    df_results["Harness_Faithfulness"].mean(),
    df_results["Harness_Relevance"].mean(),
    df_results["Harness_TrajScore"].mean(),
    df_results["Harness_Composite"].mean()
]

x = range(len(metrics))
width = 0.35
plt.bar([i - width/2 for i in x], naive_scores, width=width, label="Naive ReAct", color="#ff7675")
plt.bar([i + width/2 for i in x], harness_scores, width=width, label="Middleware Harnessed", color="#00b894")
plt.ylabel("Score (0.0 ~ 1.0)", fontsize=11)
plt.title("RAG Quality & Trajectory Metric Comparison", fontsize=12, fontweight="bold")
plt.xticks(x, metrics, fontsize=10)
plt.ylim(0, 1.1)
plt.legend(loc="upper right")
plt.grid(axis="y", linestyle="--", alpha=0.7)

# 2. Composite Score Trend Across 10 Golden Cases (Line Plot)
plt.subplot(1, 2, 2)
plt.plot(df_results["ID"], df_results["Naive_Composite"], marker="o", linewidth=2, color="#d63031", label="Naive ReAct")
plt.plot(df_results["ID"], df_results["Harness_Composite"], marker="s", linewidth=2, color="#0984e3", label="Middleware Harnessed")
plt.xlabel("Golden Case ID", fontsize=11)
plt.ylabel("Composite Score (0.0 ~ 1.0)", fontsize=11)
plt.title("10 Golden Benchmark Cases: Composite Score Trend", fontsize=12, fontweight="bold")
plt.xticks(rotation=45, fontsize=9)
plt.ylim(0, 1.1)
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
chart_path = os.path.join(PROJECT_ROOT, "artifacts", "rag_benchmark_comparison.png")
os.makedirs(os.path.dirname(chart_path), exist_ok=True)
plt.savefig(chart_path, dpi=200)
plt.show()

print(f"✅ Comparison chart saved successfully: {chart_path}")


## 7. 🎓 엔터프라이즈 에이전틱 RAG 실무 핵심 요약 (Key Engineering Takeaways)

---

### 🏆 오늘 우리가 구축한 아키텍처의 차별점

1. **에이전트 본체와 하네스의 완벽한 분리 (Separation of Concerns)**
   * 복잡한 검증 루프와 자가 수정 로직을 ReAct 그래프 내부에 하드코딩하지 않고, **LangChain `AgentMiddleware` 표준 인터페이스(`wrap_tool_call`, `after_agent`)**로 모듈화하여 탈부착이 가능하게 만들었습니다.
2. **2-Stage 자가 수정(Self-Correction) 메커니즘**
   * **1단계(도구 레벨)**: `QueryRewriter`가 모호한 구어체나 오탈자 쿼리를 감지하여 즉시 1회 자동 재검색을 수행합니다.
   * **2단계(응답 레벨)**: `HallucinationGrader`가 생성된 답변의 근거(Groundedness)를 채점하여 환각 발생 시 피드백 메시지를 주입하고 스스로 정정하도록 유도합니다.
3. **과정(Trajectory)과 결과(Outcome)를 모두 감사하는 2계층 평가 하네스**
   * 최종 답변의 정확도(RAGAS)뿐만 아니라, **에이전트가 어떤 도구를 골랐고, 어떤 쿼리를 날렸으며, 헛돌지 않고 최소 스텝으로 해결했는지(과정)**를 함께 정량 감사할 수 있는 프로덕션 평가 인프라를 완성했습니다.

---

### 🚀 프로덕션 배포 및 웹 UI 연동
완성된 에이전트 코드는 `app/agents/corrective_rag_agent.py`에 등록되어 있으며, 다음 명령어로 Streamlit 웹 UI에서 즉시 대화형으로 테스트할 수 있습니다:

```bash
# Streamlit 웹 UI 가동
streamlit run app/ui.py
```
* 웹 브라우저(`http://localhost:8501`)에서 **"corrective_rag_agent"**를 선택하여 미들웨어 하네스의 실시간 자가 수정 궤적을 확인해 보세요!
